# Exact Knapsack (ILP) Team Formation — Using Uploaded Skill Datasets

This notebook implements an **exact** team selection solver using a **0–1 Integer Linear Program (ILP)**:

- Choose up to **K** researchers (team size constraint)
- Maximize **weighted coverage** of the proposal’s required skills
- Uses uploaded datasets:
  - `/mnt/data/large_researcher_skills.csv`
  - `/mnt/data/large_proposal_skills.csv`

It will:
1. Load & parse skills into Python sets  
2. Build **IDF** skill weights (rarity) from researchers  
3. Solve an **exact ILP** per proposal using `PuLP` (CBC solver)  
4. Output teams + coverage + goodness and optionally save results

> If `pulp` is missing, the notebook provides a small **exact brute-force fallback** for tiny candidate pools (useful for quick testing).


In [17]:

# === 1) SETUP ===
import pandas as pd
import numpy as np
import ast
import math
import random
from collections import Counter
import os
import datetime

RESEARCHER_SKILLS_PATH = "../data/input_data/Set_4/large_researcher_skills.csv"
PROPOSAL_SKILLS_PATH   = "../data/input_data/Set_4/large_proposal_skills.csv"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Paths:")
print(" -", RESEARCHER_SKILLS_PATH)
print(" -", PROPOSAL_SKILLS_PATH)


Paths:
 - ../data/input_data/Set_4/large_researcher_skills.csv
 - ../data/input_data/Set_4/large_proposal_skills.csv


In [18]:

# === 2) LOAD DATASETS ===
researchers_df = pd.read_csv(RESEARCHER_SKILLS_PATH)
proposals_df   = pd.read_csv(PROPOSAL_SKILLS_PATH)

print("Researchers:", researchers_df.shape)
print("Proposals:  ", proposals_df.shape)

display(researchers_df.head(3))
display(proposals_df.head(3))


Researchers: (2000, 3)
Proposals:   (500, 3)


,Unnamed: 0,researcher_name,skills
0,0,Carmen Meyer,"{'ceramics', 'artificial intelligence', 'neuro..."
1,1,Danielle Rodriguez PhD,"{'project management', 'graph theory', 'biolog..."
2,2,Tyler Hill DVM,"{'artificial intelligence', 'deep learning', '..."


,Unnamed: 0,nsf_proposal_links_v0,skills
0,0,https://www.nsf.gov/pubs/2026/nsf26000/nsf2600...,"{'python', 'optimization', 'deep learning', 's..."
1,1,https://www.nsf.gov/pubs/2026/nsf26001/nsf2600...,"{'data science', 'python', 'deep learning', 'n..."
2,2,https://www.nsf.gov/pubs/2026/nsf26002/nsf2600...,"{'structural health monitoring', 'machine lear..."


In [19]:

# === 3) PARSE SKILL SETS ===
def parse_skill_set(x):
    if pd.isna(x):
        return set()
    if isinstance(x, (set, list, tuple)):
        return set(x)
    try:
        val = ast.literal_eval(x)
        return set(val) if isinstance(val, (set, list, tuple)) else set()
    except Exception:
        s = str(x).strip()
        s = s.strip("{}")
        parts = [p.strip().strip("'").strip('"') for p in s.split(",") if p.strip()]
        return set(parts)

# Expected columns (based on uploaded files)
RESEARCHER_NAME_COL = "researcher_name"
PROPOSAL_LINK_COL   = "nsf_proposal_links_v0"
SKILLS_COL          = "skills"

researchers_df["skill_set"] = researchers_df[SKILLS_COL].apply(parse_skill_set)
proposals_df["skill_set"]   = proposals_df[SKILLS_COL].apply(parse_skill_set)

# Build maps
researcher_skills = dict(zip(researchers_df[RESEARCHER_NAME_COL], researchers_df["skill_set"]))
proposal_skills   = dict(zip(proposals_df[PROPOSAL_LINK_COL], proposals_df["skill_set"]))

all_researchers = list(researcher_skills.keys())
all_proposals   = list(proposal_skills.keys())

print("Parsed.")
print("Researchers in map:", len(all_researchers))
print("Proposals in map:  ", len(all_proposals))
print("Example researcher skill count:", len(researcher_skills[all_researchers[0]]))
print("Example proposal skill count:  ", len(proposal_skills[all_proposals[0]]))


Parsed.
Researchers in map: 1968
Proposals in map:   500
Example researcher skill count: 4
Example proposal skill count:   4


In [20]:

# === 4) BUILD SKILL RARITY WEIGHTS (IDF) ===
# weight(s) = log(R / (freq(s)+1)) + 1
skill_counts = Counter()
for r, skills in researcher_skills.items():
    skill_counts.update(skills)

R = len(researcher_skills)
DEFAULT_W = 1.0

skill_weights = {s: (math.log(R / (c + 1)) + 1) for s, c in skill_counts.items()}

print("Top common skills:")
for s, c in skill_counts.most_common(10):
    print(f"  {s:35s} {c:5d}  w={skill_weights.get(s, DEFAULT_W):.3f}")

print("\nSome rare skills:")
for s, c in sorted(skill_counts.items(), key=lambda x: x[1])[:10]:
    print(f"  {s:35s} {c:5d}  w={skill_weights.get(s, DEFAULT_W):.3f}")


Top common skills:
  logistics                             228  w=3.151
  applied mathematics                   225  w=3.164
  biophysics                            223  w=3.173
  software engineering                  222  w=3.178
  business                              221  w=3.182
  python                                220  w=3.187
  geotechnical engineering              219  w=3.191
  optimization                          218  w=3.196
  robotics                              216  w=3.205
  economics                             216  w=3.205

Some rare skills:
  pedagogy                              176  w=3.409
  structural health monitoring          179  w=3.392
  polymers                              180  w=3.386
  mathematics                           180  w=3.386
  artificial intelligence               184  w=3.364
  climate change                        186  w=3.354
  material science                      187  w=3.348
  nanotechnology                        189  w=3.338
  educat

In [21]:
import itertools

def solve_exact_bruteforce_team(p_link, K=8, candidates=None, max_candidates_for_bruteforce=25):
    req = proposal_skills.get(p_link, set())
    if not req:
        return []

    if candidates is None:
        candidates = [r for r in all_researchers if researcher_skills[r].intersection(req)]

    if len(candidates) > max_candidates_for_bruteforce:
        raise ValueError(
            f"Exact brute-force is infeasible with {len(candidates)} candidates. "
            f"Set candidate_limit <= {max_candidates_for_bruteforce} for an exact run "
            f"or install PuLP/OR-Tools for ILP."
        )

    best_team, best_obj = [], -1.0

    for k in range(1, min(K, len(candidates)) + 1):
        for combo in itertools.combinations(candidates, k):
            covered = set().union(*[researcher_skills[r].intersection(req) for r in combo])
            obj = sum(skill_weights.get(s, DEFAULT_W) for s in covered)
            if obj > best_obj:
                best_obj, best_team = obj, list(combo)

    return best_team


def solve_exact_ilp_team(p_link, K=8, candidate_limit=25):
    req = proposal_skills.get(p_link, set())
    if not req:
        return [], {"status": "no_req_skills", "objective": 0.0, "covered": set(), "req": set(), "candidates": []}

    # build candidates
    candidates = [r for r in all_researchers if researcher_skills[r].intersection(req)]

    # enforce candidate_limit to keep it exact
    if candidate_limit is not None and len(candidates) > candidate_limit:
        def cand_score(r):
            return sum(skill_weights.get(s, DEFAULT_W) for s in researcher_skills[r].intersection(req))
        candidates = sorted(candidates, key=cand_score, reverse=True)[:candidate_limit]

    if not candidates:
        return [], {"status": "no_candidates", "objective": 0.0, "covered": set(), "req": req, "candidates": []}

    # PuLP missing -> exact brute force on the (limited) candidates
    team = solve_exact_bruteforce_team(p_link, K=K, candidates=candidates, max_candidates_for_bruteforce=candidate_limit)
    covered = set().union(*[researcher_skills[r].intersection(req) for r in team]) if team else set()
    obj = sum(skill_weights.get(s, DEFAULT_W) for s in covered)

    return team, {"status": "exact_bruteforce_topN", "objective": obj, "covered": covered, "req": req, "candidates": candidates}


In [25]:
# ==========================================
# 0) IMPORT M1 (make sure M1.py is reachable)
# ==========================================
import os, sys
import M1
# If you get ModuleNotFoundError, add the folder that contains M1.py:
# sys.path.append("/path/to/folder/with/M1.py")
# import M1


# ==========================================
# === 6) GOODNESS + COVERAGE HELPERS ===
# ==========================================
def coverage_pct(req, covered):
    return 100.0 * len(covered) / max(1, len(req))


def ultra_goodness(p_link, team):
    """
    Uses your paper's metric:
    M1.apply_ultra_metric(req_skills, team, pseudo_skills_map)

    Here:
    - req_skills = proposal_skills[p_link]
    - pseudo_skills_map = researcher_skills (already your final skills)
    """
    return M1.apply_ultra_metric(proposal_skills.get(p_link, set()), team, researcher_skills)


# ==========================================
# === 7) DEMO: RUN EXACT SOLVER ON ONE PROPOSAL ===
# ==========================================
demo_p = all_proposals[0]
K = 8

team, dbg = solve_exact_ilp_team(demo_p, K=K)

req = dbg.get("req", proposal_skills.get(demo_p, set()))
covered = dbg.get(
    "covered",
    set().union(*[researcher_skills[r].intersection(req) for r in team]) if team else set()
)

goodness_ultra = ultra_goodness(demo_p, team)

print("Proposal:", demo_p)
print("K:", K)
print("Solver status:", dbg.get("status"))
print("Candidate count:", len(dbg.get("candidates", [])))
print("Team size:", len(team))
print("Team:", team)
print("Req skills:", len(req))
print("Covered skills:", len(covered))
print("Coverage %:", round(coverage_pct(req, covered), 2))
print("Goodness (M1 ultra metric):", round(goodness_ultra, 4))
print("Objective (raw weighted covered sum):", round(dbg.get("objective", 0.0), 4))


# ==========================================
# === 8) RUN FOR A SUBSET (START SMALL), THEN SCALE UP ===
# ==========================================
def run_exact_for_proposals(proposal_links, K=8, candidate_limit=None):
    rows = []
    for idx, p_link in enumerate(proposal_links):
        team, dbg = solve_exact_ilp_team(p_link, K=K, candidate_limit=candidate_limit)

        req = dbg.get("req", proposal_skills.get(p_link, set()))
        covered = dbg.get(
            "covered",
            set().union(*[researcher_skills[r].intersection(req) for r in team]) if team else set()
        )

        goodness_ultra = ultra_goodness(p_link, team)

        rows.append({
            "proposal_link": p_link,
            "team_size": len(team),
            "team": team,
            "req_skills": len(req),
            "covered_skills": len(covered),
            "coverage_pct": coverage_pct(req, covered),
            "goodness_ultra": goodness_ultra,
            "objective": dbg.get("objective", np.nan),
            "solver_status": dbg.get("status", "unknown"),
            "candidate_count": len(dbg.get("candidates", [])),
            "candidate_limit": candidate_limit
        })

        if (idx + 1) % 25 == 0:
            print(f"Processed {idx+1}/{len(proposal_links)} proposals...")

    return pd.DataFrame(rows)


subset = all_proposals  # change to all_proposals for full run
results_df = run_exact_for_proposals(subset, K=8, candidate_limit=25)

display(results_df.head())


Proposal: https://www.nsf.gov/pubs/2026/nsf26000/nsf26000.htm
K: 8
Solver status: exact_bruteforce_topN
Candidate count: 25
Team size: 1
Team: ['Hayley Gutierrez']
Req skills: 4
Covered skills: 4
Coverage %: 100.0
Goodness (M1 ultra metric): 0.4
Objective (raw weighted covered sum): 12.8266
Processed 25/500 proposals...
Processed 50/500 proposals...
Processed 75/500 proposals...
Processed 100/500 proposals...
Processed 125/500 proposals...
Processed 150/500 proposals...
Processed 175/500 proposals...
Processed 200/500 proposals...
Processed 225/500 proposals...
Processed 250/500 proposals...
Processed 275/500 proposals...
Processed 300/500 proposals...
Processed 325/500 proposals...
Processed 350/500 proposals...
Processed 375/500 proposals...
Processed 400/500 proposals...
Processed 425/500 proposals...
Processed 450/500 proposals...
Processed 475/500 proposals...
Processed 500/500 proposals...


,proposal_link,team_size,team,req_skills,covered_skills,coverage_pct,goodness_ultra,objective,solver_status,candidate_count,candidate_limit
0,https://www.nsf.gov/pubs/2026/nsf26000/nsf2600...,1,[Hayley Gutierrez],4,4,100.0,0.400000,12.826566,exact_bruteforce_topN,25,25
1,https://www.nsf.gov/pubs/2026/nsf26001/nsf2600...,2,"[Robert Smith, Jerry Mayo]",6,6,100.0,0.445833,19.463397,exact_bruteforce_topN,25,25
2,https://www.nsf.gov/pubs/2026/nsf26002/nsf2600...,2,"[Marissa Flores, Kendra Meadows]",4,4,100.0,0.487500,13.176254,exact_bruteforce_topN,25,25
3,https://www.nsf.gov/pubs/2026/nsf26003/nsf2600...,2,"[Brandon Campos, Jennifer Keller]",4,4,100.0,0.487500,13.179049,exact_bruteforce_topN,25,25
4,https://www.nsf.gov/pubs/2026/nsf26004/nsf2600...,1,[Elizabeth Keller],4,4,100.0,0.400000,13.133823,exact_bruteforce_topN,25,25


In [27]:

# === 9) SAVE RESULTS ===
OUT_DIR = "../data/output_data"
os.makedirs(OUT_DIR, exist_ok=True)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
out_csv = f"{OUT_DIR}/exact_knapsack_results_{len(results_df)}_{timestamp}.csv"

results_df.to_csv(out_csv, index=False)

print("Saved:", out_csv)


Saved: ../data/output_data/exact_knapsack_results_500_20260204_231232.csv
